In [5]:
print(df.columns)

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')


In [1]:
import pandas as pd
import json
import os
import sys

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

from src.models import get_base_models, train_models
from src.ensemble import weighted_average, blending_train, blending_predict
from src.tuning import find_best_weights

In [2]:
os.makedirs("outputs", exist_ok=True)

In [3]:
df = pd.read_csv("data/data.csv")

In [13]:
X = df.drop("price", axis=1)
y = df["price"]

In [14]:
X = pd.get_dummies(X, drop_first=True)

In [15]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Further split for blending
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

In [16]:
# Train Base Models
models = get_base_models()
models = train_models(models, X_train, y_train)

In [17]:
# Weighted Averaging (Optimized)
best_weights, best_score = find_best_weights(models, X_val, y_val)

weighted_preds = weighted_average(models, X_test, best_weights)

In [18]:
# Blending
meta_model = LinearRegression()
meta_model = blending_train(meta_model, models, X_val, y_val)

blended_preds = blending_predict(meta_model, models, X_test)

In [19]:
# Evaluation
def rmse(y_true, preds):
    return mean_squared_error(y_true, preds) ** 0.5

weighted_rmse = rmse(y_test, weighted_preds)
blended_rmse = rmse(y_test, blended_preds)

print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Blended RMSE: {blended_rmse:.4f}")

Weighted RMSE: 1325232.9564
Blended RMSE: 1324262.8392


In [20]:
# Save Predictions
pred_df = pd.DataFrame({
    "Actual": y_test.values,
    "Weighted_Predicted": weighted_preds,
    "Blended_Predicted": blended_preds
})

pred_df.to_csv("outputs/predictions.csv", index=False)

In [21]:
# Save Metrics
metrics = {
    "weighted_rmse": float(weighted_rmse),
    "blended_rmse": float(blended_rmse),
    "best_weights": best_weights.tolist()
}

with open("outputs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)


print("Pipeline completed successfully!")

Pipeline completed successfully!
